# PRE-DATA PROCESSING
DRUG DATASET: BASHI-2024

In [1]:
import pandas
import os

In [6]:
# DIRECTORIES
notebook_dir = os.getcwd()
base_dir = os.path.dirname(notebook_dir) # the notebook is in a subdirectory of the base directory
input_dir = os.path.join(base_dir, 'data', 'input', 'oneil')
print(f"Base directory: {base_dir}")
print(f"Input directory: {input_dir}")

Base directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\DREXPA
Input directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\DREXPA\data\input\oneil


In [9]:
# Load the oneilson dataset from xls file
drugdata_file = input_dir + '/oneils_dataset.xls'
drugdata_df = pandas.read_excel(drugdata_file)
drugdata_df.columns

c:\Users\viviamsb\AppData\Local\anaconda3\envs\drexpa\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Index(['BatchID', 'cell_line', 'drugA_name', 'drugA Conc (µM)', 'drugB_name',
       'drugB Conc (µM)', 'combination_name', 'viability1', 'viability2',
       'viability3', 'viability4', 'mu/muMax', 'X/X0'],
      dtype='str')

In [10]:
# UNIQUE TRIPLETS IN DATASET
triplets = drugdata_df[['cell_line', 'drugA_name', 'drugB_name']].copy()
triplets = triplets.drop_duplicates()
triplets = triplets.reset_index(drop=True)
print(f'Total unique (cell_line, drugA, drugB) triplets in dataset: {len(triplets)}')

# UNIQUE DRUG COMBINATIONS IN DATASET
drug_combinations = drugdata_df[['drugA_name', 'drugB_name']].copy()
drug_combinations = drug_combinations.drop_duplicates()
drug_combinations = drug_combinations.reset_index(drop=True)
print(f'Total unique drug combinations in dataset: {len(drug_combinations)}')

Total unique (cell_line, drugA, drugB) triplets in dataset: 22737
Total unique drug combinations in dataset: 583


In [ ]:
# CLEAN CELL LINE NAMES
drugdata_df['cell_line'] = drugdata_df['cell_line'].str.upper()
# Remove '-' or '+' from cell line names
drugdata_df['cell_line'] = drugdata_df['cell_line'].str.replace('-', '', regex=False)
drugdata_df['cell_line'] = drugdata_df['cell_line'].str.replace('+', '', regex=False)

In [ ]:
# UNIQUE DRUGS IN O'NEILS DATASET
oneils_drugs = drugdata_df[['drugA_name', 'drugB_name']].copy()
oneils_drugs['drugA_name'] = oneils_drugs['drugA_name'].str.upper()
oneils_drugs['drugB_name'] = oneils_drugs['drugB_name'].str.upper()

# Make list of total drugs in oneils dataset
oneils_drug_list = sorted(set(oneils_drugs['drugA_name'].tolist() + oneils_drugs['drugB_name'].tolist()))
print(f'Total unique drugs in ONeils dataset: {len(oneils_drug_list)}')

# Save list as dataframe of single column 'drug_name'
oneils_drug_df = pandas.DataFrame(oneils_drug_list, columns=['drug_name'])
oneils_drug_df.to_csv('oneils_drug_names.txt', index=False, header=True)

Total unique drugs in ONeils dataset: 38


In [ ]:
# UNIQUE CELL LINES IN O'NEILS DATASET
oneils_celllines = drugdata_df['cell_line'].unique().tolist()
oneils_celllines.sort()
print(f'Total unique cell lines in ONeils dataset: {len(oneils_celllines)}')

Total unique cell lines in ONeils dataset: 39


In [ ]:
# UNIQUE TRIPLETS IN O'NEILS DATASET
oneils_triplets = drugdata_df[['cell_line', 'drugA_name', 'drugB_name']].copy()
oneils_triplets = oneils_triplets.drop_duplicates()
oneils_triplets = oneils_triplets.reset_index(drop=True)
print(f'Total unique (cell_line, drugA, drugB) triplets in ONeils dataset: {len(oneils_triplets)}')

Total unique (cell_line, drugA, drugB) triplets in ONeils dataset: 22737


In [17]:
# Load the DrugCombDB dataset
drugcombdb_file = 'Syner&Antag_bliss.csv'
drugcombdb_data = pandas.read_csv(drugcombdb_file)

# CLEAN CELL LINE NAMES IN DrugCombDB
drugcombdb_data['cell_line'] = drugcombdb_data['cell_line'].str.upper()
# Remove '-' or '+' from cell line names
drugcombdb_data['cell_line'] = drugcombdb_data['cell_line'].str.replace('-', '', regex=False)
drugcombdb_data['cell_line'] = drugcombdb_data['cell_line'].str.replace('+', '', regex=False)

drugcombdb_data

,ID,Drug1,Drug2,cell_line,Bliss,classification
0,1,5-FU,ABT-888,A2058,6.260,synergy
1,2,5-FU,ABT-888,A2058,12.330,synergy
2,3,5-FU,ABT-888,A2058,11.660,synergy
3,4,5-FU,ABT-888,A2058,5.150,synergy
4,5,5-FU,AZD1775,A2058,15.770,synergy
...,...,...,...,...,...,...
280844,280845,Ivachtin,ibrutinib,TMD8,-4.509,antagonism
280845,280846,PAC-1,ibrutinib,TMD8,3.411,synergy
280846,280847,Necrostatin-1,ibrutinib,TMD8,6.138,synergy
280847,280848,NCGC00262398,ibrutinib,TMD8,-5.464,antagonism


In [18]:
# Create a filtered and aggregated version of drugcombdb_data
# that matches the unique triplets from oneils_data

# Rename columns to match for comparison
drugcombdb_filtered = drugcombdb_data[['Drug1', 'Drug2', 'cell_line', 'Bliss']].copy()
drugcombdb_filtered.columns = ['drugA_name', 'drugB_name', 'cell_line', 'bliss']

# Normalize drug names to uppercase for consistent matching
drugcombdb_filtered['drugA_name'] = drugcombdb_filtered['drugA_name'].str.upper()
drugcombdb_filtered['drugB_name'] = drugcombdb_filtered['drugB_name'].str.upper()
oneils_triplets['drugA_name'] = oneils_triplets['drugA_name'].str.upper()
oneils_triplets['drugB_name'] = oneils_triplets['drugB_name'].str.upper()

# Merge with oneils_triplets unique triplets to filter
merged = drugcombdb_filtered.merge(
    oneils_triplets[['cell_line', 'drugA_name', 'drugB_name']],
    on=['cell_line', 'drugA_name', 'drugB_name'],
    how='inner'
)

# Group by triplet and calculate mean Bliss value
mean_bliss = merged.groupby(['cell_line', 'drugA_name', 'drugB_name'])['bliss'].mean().reset_index()
mean_bliss.columns = ['cell_line', 'drugA_name', 'drugB_name', 'mean_bliss']

In [19]:
# UNIQUE DRUG NAMES IN MERGED DATASET
merged_drugs = pandas.unique(
    merged[['drugA_name', 'drugB_name']].values.ravel()
)
merged_drugs.sort()
print(f'Total unique drugs in merged dataset: {len(merged_drugs)}')

# UNIQUE CELL LINES IN MERGED DATASET
merged_celllines = merged['cell_line'].unique().tolist()
merged_celllines.sort()
print(f'Total unique cell lines in merged dataset: {len(merged_celllines)}')

# Find missing cell lines from oneils_triplets
missing_celllines = set(oneils_triplets['cell_line'].unique()) - set(merged_celllines)
print(f'Missing cell lines from merged dataset: {missing_celllines}')

Total unique drugs in merged dataset: 38
Total unique cell lines in merged dataset: 39
Missing cell lines from merged dataset: set()


In [4]:
# Load tissue information
tissue_info_file = 'tissue_cline.csv'
tissue_info = pandas.read_csv(tissue_info_file)

print(f'Total cell lines in tissue info: {tissue_info["cell"].nunique()}')
print(f'Total tissues in tissue info: {tissue_info["Tissue"].nunique()}')
print(f'Tissue types: {tissue_info["Tissue"].unique().tolist()}')

# tissue_clines_list = tissue_info['cell'].unique().tolist()

# # Find missing cell lines in tissue info
# missing_clines = set(merged_celllines) - set(tissue_clines_list)
# print(f'Cell lines missing in tissue info: {missing_clines}')


Total cell lines in tissue info: 39
Total tissues in tissue info: 6
Tissue types: ['skin', 'ovary', 'lung', 'large intestine', 'breast', 'prostate']


In [ ]:
# Map each cell line to its tissue type
tissue_dict = dict(zip(tissue_info['cell'], tissue_info['Tissue']))
mean_bliss['tissue'] = mean_bliss['cell_line'].map(tissue_dict)
mean_bliss.to_csv('oneils_bliss.csv', index=False)

,cell_line,drugA_name,drugB_name,mean_bliss,tissue
0,A2058,5-FU,ABT-888,8.850000,skin
1,A2058,5-FU,AZD1775,11.207500,skin
2,A2058,5-FU,BEZ-235,3.837500,skin
3,A2058,5-FU,BORTEZOMIB,3.896667,skin
4,A2058,5-FU,DASATINIB,7.570000,skin
...,...,...,...,...,...
20330,ZR751,ZOLINZA,SN-38,-4.125000,breast
20331,ZR751,ZOLINZA,SORAFENIB,3.390000,breast
20332,ZR751,ZOLINZA,SUNITINIB,-8.680000,breast
20333,ZR751,ZOLINZA,TEMOZOLOMIDE,6.805000,breast
